In [1]:
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)
from src.llm_word_logprobs import *

import mne

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
import seaborn as sns

import math
import re
from collections import namedtuple
from typing import Callable

from sklearn.model_selection import LeaveOneOut, KFold
from sklearn.linear_model import Ridge
import torch
from transformers import BertTokenizerFast, AutoModelForCausalLM

from scipy.stats import ttest_1samp
from statsmodels.stats.multitest import fdrcorrection

from IPython.display import clear_output

/Users/jowanglin/regression-based_ERP/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [18]:
DATA_DIR = "/Users/jowanglin/Word-Position-Effect_BLP-lab/Preprocessing/ica-data"
RAW_FILE_PATH = "SUBJ020/subj020.set"
EPOCHS_FILE_PATH = "SUBJ020/subj020_chan_rr_elist_bin_epbaseline_AD_BPdot1-30.set"

SOA = 0.517    # in seconds
DOWNSAMPLE_SFREQ = 100.0  # downsample to 100.0 Hz
N_WORDS = 7
N_TRIALS = 208

RIDGE_TOL = 1e-5
MAX_ITER = 20000
RANDOM_STATE = 42

In [4]:
def to_num_str(cell):
    if isinstance(cell, str):
        if "boundary" in cell:
            return "-1"
        else:
            return re.findall(r"\d+", cell)[0]
    else:
        return cell

raw = mne.io.read_raw_eeglab(f"{DATA_DIR}/{RAW_FILE_PATH}", preload=True, verbose=False)
df_annot = pd.DataFrame(raw.annotations)
df_annot = df_annot.map(to_num_str)
display(df_annot)

/var/folders/n3/nqqvqq1n0jx25dfd3nyxjxmh0000gn/T/ipykernel_87866/53527967.py:10: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = mne.io.read_raw_eeglab(f"{DATA_DIR}/{RAW_FILE_PATH}", preload=True, verbose=False)


,onset,duration,description,orig_time,extras
0,0.000,0.000,-1,None,{}
1,34.625,0.001,254,None,{}
2,35.209,0.001,1,None,{}
3,45.816,0.001,254,None,{}
4,46.337,0.001,251,None,{}
...,...,...,...,...,...
3223,3239.549,0.001,253,None,{}
3224,3241.064,0.001,1,None,{}
3225,3242.796,0.001,254,None,{}
3226,3244.315,0.001,1,None,{}


In [5]:
def revise_annot(df_annot: pd.DataFrame, *,
                 item_codes: list, condition_codes: list, non_final: str) -> mne.Annotations:
    desc = list(df_annot["description"])
    desc_revised = desc.copy()

    for i, d in enumerate(desc):
        if d in item_codes:
            if desc[i+1] == non_final:
                step = 0
        elif d == non_final:
            step += 1
            desc_revised[i] = "w" + str(step)
        
        elif d in condition_codes:
            desc_revised[i-step: i] = [s + "/" + d for s in desc_revised[i-step: i]]

    annot = mne.Annotations(onset=df_annot["onset"], duration=df_annot["duration"], description=desc_revised)
    return annot
            
item_codes = list(df_annot["description"].value_counts()[df_annot["description"].value_counts() == 2].index)
item_codes.append("1")
condition_codes = list(df_annot["description"].value_counts()[df_annot["description"].value_counts() == 28].index)
non_final = "255"

revised_annot = revise_annot(df_annot, item_codes=item_codes, condition_codes=condition_codes, non_final=non_final)
raw.set_annotations(revised_annot)
df_revised_annot = pd.DataFrame(raw.annotations)
display(df_revised_annot.iloc[3:23])

,onset,duration,description,orig_time,extras
3,45.816,0.001,254,None,{}
4,46.337,0.001,251,None,{}
5,47.354,0.001,105,None,{}
6,47.871,0.001,w1/244,None,{}
7,48.404,0.001,w2/244,None,{}
8,48.921,0.001,w3/244,None,{}
9,49.437,0.001,w4/244,None,{}
10,49.954,0.001,w5/244,None,{}
11,50.471,0.001,w6/244,None,{}
12,50.987,0.001,w7/244,None,{}


### Get word log probabilities

In [3]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-chinese")
model = AutoModelForCausalLM.from_pretrained("ckiplab/gpt2-base-chinese") 

Loading weights: 100%|██████████| 149/149 [00:00<00:00, 40777.20it/s]
[transformers] GPT2LMHeadModel LOAD REPORT from: ckiplab/gpt2-base-chinese
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
file_dir = "/Users/jowanglin/regression-based_ERP/data/stimuli"
file_name = "CRYSTAL_master-sheet.xlsx"
sheet_name = "Overall"
frame_col = "Sentence_frame"
word_cols = [f"W{i}" for i in range(1,13)]
prompt = "通順嗎？"

sentences, split_sentences = get_sentences_and_splits(file_dir,
                                                      file_name,
                                                      sheet_name,
                                                      frame_col=frame_col,
                                                      word_cols=word_cols,
                                                      prompt=prompt)
    
ids, tokens, token_logprobs = get_llm_token_logprobs(sentences,
                                                     model, 
                                                     tokenizer,
                                                     max_length=MAX_LENGTH)
    
pad_token, cls_token, sep_token = tokenizer.pad_token, tokenizer.cls_token, tokenizer.sep_token
pad_id, cls_id, sep_id = tokenizer.pad_token_id, tokenizer.cls_token_id, tokenizer.sep_token_id
    
word_logprobs = get_word_logprobs(tokens=tokens,
                                  split_sentences=split_sentences,
                                  ids=ids,
                                  token_logprobs=token_logprobs,
                                  special_tokens=(pad_token, cls_token, sep_token),
                                  special_token_ids=(pad_id, cls_id, sep_id),
                                  other_tokens=OTHER_TOKENS)

[UNK] found in sentence index 21, token index 24:
  ['她', '家', '就', '在', '學', '校', '旁', '還', '每', '次', '遲', '到', '就', '說', '因', '為', '塞', '車', '，', '其', '實', '都', '是', '胡', '[UNK]', '。'] 
  ['她家', '就在', '學校旁', '還每次', '遲到', '就說', '因為', '塞車，', '其實', '都是', '胡謅。']
[UNK] found in sentence index 240, token index 21:
  ['去', '完', '角', '質', '塗', '上', '乳', '液', '後', '，', '皮', '膚', '摸', '起', '來', '像', '打', '蠟', '一', '般', '胡', '[UNK]', '。'] 
  ['去完', '角質', '塗上', '乳液後，', '皮膚', '摸起來', '像打蠟', '一般', '胡謅。']
sentence indices where alignment failed: [ 21 240]


In [5]:
word_counts = [len(s) for s in split_sentences]
word_logprobs_counts = [len(p) for p in word_logprobs]
not_aligned = np.where(np.array(word_counts) - np.array(word_logprobs_counts) != 0)[0]
print(not_aligned)

[ 21 240]


In [ ]:
### TODO: handle list 1 & 2 (see master sheet)

In [6]:
print(len(word_logprobs))

416


### Deconvolution with SSP

In [ ]:
def build_design(*, soa: float, sfreq: float,
                 n_words: int, n_trials: int,
                 response_lag: float,
                 rank_tol: float=1e-3,
                 intercept: bool=True,
                 **predictors):
    if predictors:
        for p in predictors.values():
            assert p.shape == (n_trials, n_words), "Each predictor must be a numpy array of shape (n_trials, n_words)"

    DeconvDesignMatrix = namedtuple("DeconvDesignMatrix",
                                    ["predictors", "n_lags", "rank", "singular_vals", "condition_num"])
    
    if intercept:
        predictors["intercept"] = np.ones((n_trials, n_words), dtype=float)

    n_time_samples_per_trial = int(round(soa * n_words * sfreq))
    n_time_samples = int(n_time_samples_per_trial * n_trials) 
    n_lags = int(round(response_lag * sfreq))
    soa_samples = int(round(soa * sfreq))
    
    matrices = {name: np.zeros((n_time_samples, n_lags), dtype=float) for name in predictors}
    
    positions = range(n_words)
    for i in range(n_trials):
        for j in positions:
            valid_range = min(n_time_samples_per_trial - j * soa_samples, n_lags)
            row = i * n_time_samples_per_trial + j * soa_samples
            for m, p in zip(matrices.values(), predictors.values()):
                m[row : row + valid_range, : valid_range] += p[i, j] * np.eye((valid_range))

    X_full = np.hstack([m for m in matrices.values()])
    rank = np.linalg.matrix_rank(X_full, tol=rank_tol)
    singular_vals = np.linalg.svd(X_full, compute_uv=False)
    condition_num = np.linalg.cond(X_full)

    return DeconvDesignMatrix(matrices, n_lags, rank, singular_vals, condition_num)



word_pos_predictor = np.vstack([np.log(np.arange(1, N_WORDS+1)) for _ in range(N_TRIALS)])

### TODO
'''word_logprob_predictor = None

deconv_design_matrix = build_design(soa=SOA,
                                    sfreq=DOWNSAMPLE_SFREQ,
                                    n_words=N_WORDS, n_trials=N_TRIALS,
                                    response_lag=0.7,
                                    rank_tol=1e-2,
                                    word_pos=word_pos_predictor,
                                    word_logprobs=word_logprob_predictor)'''

    


In [ ]:
class SSP:
    def __init__(self):
        pass
    def fit_transform(self):
        pass
    def transform(self):
        pass


def tune_alpha(ridge_data: np.ndarray, *, X: np.ndarray,
               alpha_range: np.ndarray,
               ssp: Callable | None=None, info: mne.Info | None=None,
               loo: bool=False, n_splits: int | None=None,
               ridge_tol: float=1e-4, solver: str="auto", max_iter: int|None=None,
               verbose: bool=False):
    """ridge_data.shape = (n_trials, n_roi_channels, n_time_samples)
                          if doing LOO (e.g., training on 19 subjects, testing on 1), n_trials = n_subjects (but this does not work well)
       design matrix X.shape = (n_time_samples, n_predictors * n_lags)
       alpha_range in single-subject-level range (not divided by n_subj-1)"""
    ScoreResults = namedtuple("ScoreResults", ["train_scores", "train_mean", "train_std",
                                               "valid_scores", "valid_mean", "valid_std"])
    CVResults = namedtuple("CVResults", ["fold_scores", "best_mean_score", "best_alpha"])   
    
    n_trials, n_roi_channels, n_time_samples = ridge_data.shape
    assert n_time_samples == X.shape[0]

    if loo:
        splitter = LeaveOneOut()
        n_splits = n_trials
    else:
        if n_splits is None:
            n_splits = 5
        splitter = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    
    train_scores = np.empty((n_splits, len(alpha_range), n_roi_channels))
    valid_scores = np.empty((n_splits, len(alpha_range), n_roi_channels))

    for i, (train_idx, valid_idx) in enumerate(splitter.split(ridge_data)):
        if verbose:
            print(f"  FOLD {i+1}")

        y_train, y_valid = ridge_data[train_idx], ridge_data[valid_idx]  # y_train.shape = (n_train_trials, n_roi_channels, n_time_samples)

        X_train_stack = np.vstack([X for _ in train_idx])  # shape = (n_train_trials * n_time_samples, response_lag)
        y_train_stack = np.hstack([y for y in y_train])    # shape = (n_roi_channels, n_train_trials * n_time_samples)
            
        if ssp is not None:
            y_train_stack = ssp.fit_transform(y_train_stack, info=info) 
            # y_train = y_train_stack.reshape(y_train.shape)  <- not safe
            y_train = np.stack(np.split(y_train_stack, len(train_idx), axis=1), axis=0)

            y_valid_stack = np.hstack([y for y in y_valid])      
            y_valid_stack = ssp.transform(y_valid_stack, info=info)    
            # y_valid = y_valid_stack.reshape(y_valid.shape)  <- not safe    
            y_valid = np.stack(np.split(y_valid_stack, len(valid_idx), axis=1), axis=0)
        
        for j, alpha in enumerate(alpha_range):
            if verbose:
                print(f"ALPHA = {alpha}")   

            ridge = Ridge(alpha=alpha,
                          fit_intercept=False,         
                          tol=ridge_tol,
                          solver=solver,
                          positive=False,              
                          max_iter=max_iter,
                          random_state=RANDOM_STATE)
            
            for k in range(n_roi_channels):
                if verbose:
                    print(f"    Channel Index {k}")

                y_train_per_ch = y_train_stack[k, :]
                ridge.fit(X_train_stack, y_train_per_ch)
                train_scores[i, j, k] = ridge.score(X, np.mean(y_train[:, k, :], axis=0))
                valid_scores[i, j, k] = ridge.score(X, np.mean(y_valid[:, k, :], axis=0))
    
    fold_scores = {alpha: ScoreResults(train_scores[:, j, :], np.mean(train_scores[:, j, :]), np.std(train_scores[:, j, :]),
                                       valid_scores[:, j, :], np.mean(valid_scores[:, j, :]), np.std(valid_scores[:, j, :]))
                          for j, alpha in enumerate(alpha_range)}
            
    mean_scores = np.array([fold_scores[alpha].valid_mean for alpha in alpha_range])
    best_mean_score = np.max(mean_scores)
    best_alpha = alpha_range[np.argmax(mean_scores)]

    return CVResults(fold_scores, best_mean_score, best_alpha)

